# Delta Lake MERGE Implementation — Assignment 7

Dataset: Superstore (Kaggle). Environment: Databricks Free Edition (Serverless), Delta Lake Slowly Changing Dimensions(SCD) Assignment.


## Step 1: Load dataset from Volume

In [0]:
master_path = "/Volumes/workspace/default/delta_assignment/Superstore-data.csv"

df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv(master_path))

display(df)
print("Row count:", df.count())
df.printSchema()

## Step 2: Add unique Row_ID, clean numeric columns, fix column names

In [0]:
from pyspark.sql.functions import expr

df2 = (df
       .withColumn("Sales", expr("try_cast(regexp_replace(Sales, '[^0-9.]', '') as double)"))
       .withColumn("Quantity", expr("try_cast(regexp_replace(Quantity, '[^0-9]', '') as int)"))
       .withColumn("Discount", expr("try_cast(regexp_replace(Discount, '[^0-9.]', '') as double)")))

new_cols = [c.replace(" ", "_").replace("-", "_") for c in df2.columns]
df2 = df2.toDF(*new_cols)

display(df2)
df2.printSchema()

## Step 3: Save as Delta table (customer_master)

In [0]:
df2.write.format("delta").mode("overwrite").saveAsTable("customer_master")
print("Saved. Row count:", spark.table("customer_master").count())

## Step 4: Basic cleaning — handle nulls, remove duplicates

In [0]:
tbl = spark.table("customer_master")

# Check null counts per column
tbl.select([expr(f"sum(case when {c} is null then 1 else 0 end) as {c}") for c in ["Sales","Quantity","Discount","Customer_ID","City"]]).show()

In [0]:
# Drop rows with null Sales, Quantity or Customer_ID (core fields), remove duplicate rows
df_clean = tbl.dropna(subset=["Sales", "Quantity", "Customer_ID"])
df_clean = df_clean.dropDuplicates()

print("Before clean:", tbl.count(), "| After clean:", df_clean.count())

df_clean.write.format("delta").mode("overwrite").saveAsTable("customer_master")
display(spark.table("customer_master"))

## Step 5: Create incremental dataset (simulate new + updated records)

In [0]:
master_df = spark.table("customer_master")

# 90% stays as master baseline, 10% set aside as "new incoming" rows
new_rows = master_df.sample(fraction=0.1, seed=42)
remaining = master_df.subtract(new_rows)

remaining.write.format("delta").mode("overwrite").saveAsTable("customer_master")

# Simulate UPDATES: take 5 existing rows, change Sales & Quantity
update_rows = remaining.sample(fraction=0.01, seed=1).limit(5)
update_rows = (update_rows
               .withColumn("Sales", expr("Sales * 1.1"))
               .withColumn("Quantity", expr("Quantity + 1")))

# Combine new + updated rows into incremental dataset
df_incremental = new_rows.unionByName(update_rows)

df_incremental.write.format("delta").mode("overwrite").saveAsTable("customer_incremental")

print("Master rows:", spark.table("customer_master").count())
print("Incremental rows:", df_incremental.count())
display(df_incremental)

## Step 6: Apply MERGE operation (update existing + insert new)

In [0]:
from delta.tables import DeltaTable

delta_master = DeltaTable.forName(spark, "customer_master")
df_incr = spark.table("customer_incremental")

(delta_master.alias("t")
 .merge(
     df_incr.alias("s"),
     "t.Row_ID = s.Row_ID"
 )
 .whenMatchedUpdate(set={
     "Sales": "s.Sales",
     "Quantity": "s.Quantity",
     "Discount": "s.Discount"
 })
 .whenNotMatchedInsertAll()
 .execute())

print("MERGE completed.")

## Step 7: Validate results — row count, duplicates

In [0]:
df_final = spark.table("customer_master")

print("Final row count:", df_final.count())
print("Distinct Row_ID count:", df_final.select("Row_ID").distinct().count())

dup_count = (df_final.groupBy("Row_ID").count()
             .filter("count > 1").count())
print("Duplicate Row_ID groups:", dup_count)

## Step 8: Display final dataset

In [0]:
display(df_final.orderBy("Row_ID"))

## Summary

- Loaded Superstore dataset (9994 rows) from a Databricks Volume into a Delta table `customer_master`.
- Cleaned numeric fields (Sales, Quantity, Discount) using try_cast to handle malformed/empty values, and fixed invalid column names (spaces/hyphens) for Delta compatibility. The dataset's existing Row ID column was used as the unique key.
- Cleaned data: dropped rows with nulls in core fields (Sales, Quantity, Customer ID) and removed duplicate rows.
- Simulated incremental data: 10% of rows held out as "new" records, plus a few existing rows modified to simulate updates — stored as `customer_incremental`.
- Applied Delta Lake MERGE INTO: matched rows (by `Row_ID`) had Sales/Quantity/Discount updated, unmatched rows were inserted as new — all in a single atomic transaction.
- Validated: final row count (8771) equaled distinct `Row_ID` count, with zero duplicate groups, confirming a clean, consistent upsert.
- Delta Lake's MERGE replaced manual UPDATE/INSERT logic with one ACID-compliant operation, demonstrating efficient incremental processing.